In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import regex
import requests

from src.api import (
    getCirculaciones,
    getEstaciones,
    getHistoricoMOW,
    getInfoAPIs,
    getInfoToposMSE,
    getInfoToposMSEWithoutCTC,
    hacerPeticion,
)
from src.utils import guardarExcel, loadEstaciones, parallelizeFunction, time2localtime

In [ ]:
# cat_feb = pd.read_excel(
#     r"C:\Users\jose.espinosa\Downloads\NOE-20250213.xlsx",
#     sheet_name="OBJETOS",
#     header=None,
# ).drop(labels=[3], axis=1, errors="ignore")
# cat_feb_bajas = pd.read_excel(
#     r"C:\Users\jose.espinosa\Downloads\NOE-20250213.xlsx",
#     sheet_name="BAJAS",
#     header=None,
# ).drop(labels=[3], axis=1, errors="ignore")
# cat_feb["fuente"] = "feb"

# cat_oct = pd.read_excel(
#     r"C:\Users\jose.espinosa\Downloads\NOE-20241013.xlsx",
#     sheet_name="OBJETOS",
#     header=None,
# ).drop(labels=[3], axis=1, errors="ignore")
# cat_oct["fuente"] = "oct"


# teorico = pd.concat([cat_feb, cat_feb_bajas]).sort_values(by=[0, 1, 2])
# teorico["fuente"] = "feb"

# comparacion = (
#     pd.merge(cat_oct, teorico, how="outer", on=[0, 1, 2])
#     .sort_values(by=[0, 1, 2])
#     .drop_duplicates()
# )
# comparacion.columns=["Mnemónico", "Elemento", "Tipo", "FuenteOctubre", "FuenteFebrero"]
# guardarExcel(comparacion, "Diferencias_octubre_febrero.xlsx")
# # (comparacion[comparacion["fuente_y"].isna()])

In [ ]:
info_topos = getInfoToposMSE()
info_topos_sin_ctc = getInfoToposMSEWithoutCTC()

In [ ]:
with Path("data/estaciones_HMI_topo.json").open("w", encoding="utf8") as f:
    json.dump(info_topos, f, ensure_ascii=False, indent=4)
# with Path("data/estaciones_HMI_topo_sin_ctc.json").open("w", encoding="utf8") as f:
#     json.dump(info_topos_sin_ctc, f, ensure_ascii=False, indent=4)

In [ ]:
def loadEstacionesSinCTC():
    """
    Carga info de las estaciones según IHM topo
    """
    with open(r"data\estaciones_HMI_topo_pre.json", "r", encoding="utf8") as f:
        # conectores = pd.json_normalize(json.load(f), max_level=10)
        conectores = json.load(f)["pointIdsWithoutCTC"]
    cod2ctc = []
    for con in conectores:
        cod2ctc.append(
            (
                con.get("pointId").strip(),
                con.get("pointName").strip(),
                con.get("delegation").strip(),
                con.get("trustedArrival"),
                con.get("trustedDeparture"),
                con.get("trustedDepartureOrigin"),
                con.get("trustedTracks"),
            )
        )
        # cod2ctc.append((dep["pointId"], con['configCTCMetadata']['ctcName']))
    # cod2ctc = dict(cod2ctc)
    cod2ctc = (
        pd.DataFrame(
            cod2ctc,
            columns=[
                "Código",
                "Nombre",
                "Delegación",
                "LlegadaFiable",
                "SalidaFiable",
                "SalidaFiableOrigen",
                "VíasFiables",
            ],
        )
    )
    return cod2ctc

In [ ]:
def loadEstaciones():
    """
    Carga info de las estaciones según IHM topo
    """
    with open(r"data\estaciones_HMI_topo.json", "r", encoding="utf8") as f:
        # conectores = pd.json_normalize(json.load(f), max_level=10)
        conectores = json.load(f)["configCTCs"]
    cod2ctc = []
    for con in conectores:
        # print(con['configCTCMetadata']['ctcName'])
        if not con["configInterLocks"]:
            continue
        metadata = con["configCTCMetadata"]
        ctc = metadata["ctc"].strip()
        ctcName = regex.sub(r"\bCTC\b", "", metadata["ctcName"]).strip()
        tecnologo = metadata["manufacturer"]
        catalogo = metadata["catalogueId"]
        for interlock in con["configInterLocks"]:
            mnem = interlock["configInterLockMetadata"]["interlock"]
            name = interlock["configInterLockMetadata"]["interlockName"]
            for dep in interlock["configDependences"]:
                cod2ctc.append(
                    (
                        dep.get("delegation"),
                        catalogo or dep.get("catalogueId"),
                        ctc,
                        ctcName,
                        tecnologo,
                        name,
                        mnem,
                        dep.get("pointId"),
                        dep.get("pointName"),
                        dep.get("acronym"),
                    )
                )
                # cod2ctc.append((dep["pointId"], con['configCTCMetadata']['ctcName']))
    # cod2ctc = dict(cod2ctc)
    cod2ctc = (
        pd.DataFrame(
            cod2ctc,
            columns=[
                "Delegación",
                "Catálogo",
                "CTC",
                "NombreCTC",
                "Tecnólogo",
                "Enclavamiento",
                "Mnemónico",
                "Código",
                "Nombre",
                "Mnemónico_comercial",
            ],
        )
        .fillna("")
        .map(lambda x: x.strip())
    )
    return cod2ctc

In [ ]:
# En topos y no se publica (socket_in)
# Se publica y no está en catálogo
# En topo y no en catálogo (metidas a mano)
# En catálogo y no en topo

In [ ]:
estaciones = loadEstaciones()

In [ ]:
estaciones.loc[estaciones["CTC"]=="BCN", ["Mnemónico", "CTC", "Enclavamiento"]]

In [ ]:
IP = "10.251.99.100"
HOST = "topo.rail.api.elcano.operaciones.adif"
headers = {
    "host": HOST,
    "Content-Type": "application/json; charset=UTF-8",
    "Prefer": "respond-async",
}
method = "POST"
URL = "http://topo.rail.api.elcano.operaciones.adif/msetopo/download/filesInterlock"
topoId = {"interlock": "AT", "ctc": "BCN", "interlockName": "ALTAFULLA"}
data = json.dumps(topoId)
response = hacerPeticion("POST", URL, headers=headers, data=data)

In [ ]:
topo = json.loads(json.loads(response.text.strip())["topoResource"]["configFileJson"])

In [ ]:
elementos = [
    el["id"].split(".")
    for el in topo["viewCtc"]["elements"]
    if "undefined" not in el["id"]
]
elementos = [el for el in elementos if len(el) == 4]
elementos

In [ ]:
estaciones[estaciones.duplicated(subset=["Código"], keep=False)].sort_values(
    by="Código"
)

In [ ]:
guardarExcel(
    estaciones[estaciones.duplicated(subset=["Código"], keep=False)].sort_values(
        by="Código"
    ),
    "CódigosDuplicados.xlsx"
)

In [ ]:
estaciones_sin_ctc = loadEstacionesSinCTC()

In [ ]:
guardarExcel(
    pd.merge(
        estaciones[
            [
                "Código",
                "Nombre",
                "Delegación",
                "Catálogo",
                "CTC",
                "Tecnólogo",
                "Mnemónico",
            ]
        ],
        estaciones_sin_ctc[["Código", "Nombre", "Delegación"]],
        on=["Código"],
        suffixes=["", "_sin_ctc"],
    ),
    "CódigosDuplicados_CTC_noCTC_pre.xlsx",
    append_sheet=False,
)

In [ ]:
pd.json_normalize(info_topos["configCTCs"])

In [ ]:
def hacerPeticion(method: str, URL: str, headers: dict, data: dict = None):
    response = requests.request(method, URL, headers=headers, data=data)
    if response.status_code >= 400:
        print(f"Error: '{response.status_code}' en la respuesta")
        return
    if response.status_code >= 300:
        print(f"Redirección: código'{response.status_code}'")
        return
    if response.status_code > 200:
        print(f"👍: código'{response.status_code}'")
        return
    if response.status_code < 200:
        print(f"Info: código'{response.status_code}'")
        return
    if not response.encoding == "utf-8":
        response.encoding = "utf-8"
    return response

In [ ]:
IP = "10.251.99.100"
HOST = "info.api.elcano.operaciones.adif"
URL = f"http://{IP}"

headers = {
    "host": HOST,
    "Content-Type": "application/json; charset=UTF-8",
    "Prefer": "respond-async",
}

### ToposInfo
Tabla del HMI Topo

In [ ]:
# def getEstaciones():
# """
# Obtener la información disponible para el usuario de todas las estaciones desde API de circulaciones.
# """
# HOSTPATH = "/portroyalmanager/stations/allstations/"
HOSTPATH = "http://topo.rail.api.elcano.operaciones.adif/msetopo/findAllToposInfo"
response = hacerPeticion(
    "GET",
    HOSTPATH,
    headers={
        # "host": HOST,
        "Content-Type": "application/json; charset=UTF-8",
        "Prefer": "respond-async",
    },
)
info_topos = json.loads(response.text)

In [ ]:
info_topos

In [ ]:
with Path("data/estaciones_HMI_topo.json").open("w", encoding="utf8") as f:
    json.dump(info_topos, f, ensure_ascii=False, indent=4)

### Panif

In [ ]:
HOSTPATH = "/planning/findAllCirculationsPlanning"
response = hacerPeticion(
    "GET",
    URL + HOSTPATH,
    headers=headers,
)
a = json.loads(response.text)

In [ ]:
# day = "2024-11-20T01:00:00.000Z"
# pd.to_datetime(day).timestamp()
day = "2024-11-22"
points = "'17000'"
# HOSTPATH = f"/planning/{day}/stationCodes/{points}/findAllCirculationsPlanningScopeDay"
HOSTPATH = f"/planning/perScope/{points}/findAllCirculationsPlanningScope"
HOSTPATH = f"/planning/{day}/findAllCirculationsPlanningDay"
response = hacerPeticion(
    "GET",
    URL + HOSTPATH,
    headers=headers,
)
b = json.loads(response.text)

In [ ]:
URL + HOSTPATH

In [ ]:
a = pd.json_normalize(b)[
    [
        "circulationId.number",
        "circulationId.launchingDate",
        "dayTrain.commercialNumber",
        "dayTrain.line",
        "dayTrain.company",
        "dayTrain.operator",
        "dayTrain.trainType",
        # "dayTrain.commercialTrain",
        # "dayTrain.special",
        # "dayTrain.virtual",
        "dayTrain.journey.assimilateTo",
        "dayTrain.journey.assimilatedBy",
        "dayTrain.journey.steps",
        "dayTrain.journey.journeySectionsData.timeStampInMillisSections",
        "dayTrain.journey.journeySectionsData.sections",
        "dayTrain.identifier.number",
        "dayTrain.identifier.launchingDate",
        "dayTrain.allNumbers",
        # "dayTrain.journey.assimilatedBy.circulationId.number",
        # "dayTrain.journey.assimilatedBy.circulationId.launchingDate",
        # "dayTrain.journey.assimilatedBy.originConnection.pointId",
        # "dayTrain.journey.assimilatedBy.originConnection.step",
        # "dayTrain.journey.assimilatedBy.destinationConnection.pointId",
        # "dayTrain.journey.assimilatedBy.destinationConnection.step",
        # "dayTrain.journey.assimilateTo.circulationId.number",
        # "dayTrain.journey.assimilateTo.circulationId.launchingDate",
        # "dayTrain.journey.assimilateTo.originConnection.pointId",
        # "dayTrain.journey.assimilateTo.originConnection.step",
        # "dayTrain.journey.assimilateTo.destinationConnection.pointId",
        # "dayTrain.journey.assimilateTo.destinationConnection.step",
    ]
]  # .sort_values(by=["circulationId.number", "circulationId.launchingDate"])

In [ ]:
a[a["circulationId.number"] == "00194"]["dayTrain.journey.steps"].values

In [ ]:
a.sort_values(by="circulationId.number")

### API circulaciones

In [ ]:
HOSTPATH = "/mse-circulations/doc/openapi/"
response = hacerPeticion(
    "GET",
    URL + HOSTPATH,
    headers=headers,
)
info_api_circulaciones = json.loads(response.text)

In [ ]:
info_api_circulaciones

### Circulaciones

In [ ]:
HOSTPATH = "/portroyalmanager/circulationpaths/departures/traffictype/"


movimientos = []
page = 0
station_code = "18000"
while True:
    print(f"\rPágina {page}", end="", flush=True)
    data = {
        "commercialService": "BOTH",
        "commercialStopType": "BOTH",
        "page": {"pageNumber": page},
        "stationCode": station_code,
        "trafficType": "ALL",
    }
    data = json.dumps(data)

    response = hacerPeticion(
        "POST",
        URL + HOSTPATH,
        headers=headers,
        data=data,
    )
    if not response:
        break
    response = response.json()
    movimientos.extend(response["commercialPaths"])
    page += 1
    # if page == response["totalPages"]:
    #     break

In [ ]:
df_movimientos = pd.json_normalize(movimientos)[
    [
        "commercialPathInfo.commercialPathKey.commercialCirculationKey.commercialNumber",
        "commercialPathInfo.commercialPathKey.commercialCirculationKey.launchingDate",
        "commercialPathInfo.commercialPathKey.originStationCode",
        "commercialPathInfo.commercialPathKey.destinationStationCode",
        "commercialPathInfo.line",
        "commercialPathInfo.trafficType",
        "commercialPathInfo.opeProComPro.operator",
        "commercialPathInfo.opeProComPro.commercialProduct",
        "commercialPathInfo.announceableStations",
        "passthroughStep.stopType",
        "passthroughStep.announceable",
        "passthroughStep.stationCode",
        "passthroughStep.departurePassthroughStepSides.plannedTime",
        "passthroughStep.departurePassthroughStepSides.forecastedOrAuditedDelay",
        "passthroughStep.departurePassthroughStepSides.timeType",
        "passthroughStep.departurePassthroughStepSides.plannedPlatform",
        "passthroughStep.departurePassthroughStepSides.sitraPlatform",
        "passthroughStep.departurePassthroughStepSides.ctcPlatform",
        "passthroughStep.departurePassthroughStepSides.resultantPlatform",
        "passthroughStep.departurePassthroughStepSides.circulationState",
        "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalNumber",
        "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalLaunchingDate",
    ]
].rename(
    columns={
        "commercialPathInfo.commercialPathKey.commercialCirculationKey.commercialNumber": "NComercial",
        "commercialPathInfo.commercialPathKey.commercialCirculationKey.launchingDate": "FechaOrigenComercial",
        "commercialPathInfo.commercialPathKey.originStationCode": "CódigoOrigen",
        "commercialPathInfo.commercialPathKey.destinationStationCode": "CódigoDestino",
        "commercialPathInfo.line": "Línea",
        "commercialPathInfo.trafficType": "Tráfico",
        "commercialPathInfo.opeProComPro.operator": "Operador",
        "commercialPathInfo.opeProComPro.commercialProduct": "Producto",
        "commercialPathInfo.announceableStations": "EstacionesAnunciables",
        "passthroughStep.stopType": "TipoParada",
        "passthroughStep.announceable": "Anunciable",
        "passthroughStep.stationCode": "Código",
        "passthroughStep.departurePassthroughStepSides.plannedTime": "TiempoPlanificado",
        "passthroughStep.departurePassthroughStepSides.forecastedOrAuditedDelay": "Retraso",
        "passthroughStep.departurePassthroughStepSides.timeType": "Registro",
        "passthroughStep.departurePassthroughStepSides.plannedPlatform": "VíaPlanificada",
        "passthroughStep.departurePassthroughStepSides.sitraPlatform": "VíaSitra",
        "passthroughStep.departurePassthroughStepSides.ctcPlatform": "VíaCTC",
        "passthroughStep.departurePassthroughStepSides.resultantPlatform": "TipoVía",
        "passthroughStep.departurePassthroughStepSides.circulationState": "Estado",
        "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalNumber": "NTécnico",
        "passthroughStep.departurePassthroughStepSides.technicalCirculationKey.technicalLaunchingDate": "FechaOrigenTécnico",
    }
)

In [ ]:
df_movimientos[["FechaOrigenComercial", "FechaOrigenTécnico", "TiempoPlanificado"]] = (
    pd.concat(
        parallelizeFunction(
            lambda x: x.map(time2localtime, unit="ms"),
            data=np.array_split(
                df_movimientos[
                    ["FechaOrigenComercial", "FechaOrigenTécnico", "TiempoPlanificado"]
                ],
                len(df_movimientos) // np.min((len(df_movimientos), 1000)),
            ),
            show_progress=True,
            desc="Formateando fechas.",
            output="series",
        )
    )
)
df_movimientos[["NombreOrigen", "NombreDestino", "Nombre"]] = pd.concat(
    parallelizeFunction(
        lambda x: x.map(map_codigo_nombre.get),
        data=np.array_split(
            df_movimientos[["CódigoOrigen", "CódigoDestino", "Código"]],
            len(df_movimientos) // np.min((len(df_movimientos), 1000)),
        ),
        show_progress=True,
        desc="Formateando fechas.",
        output="series",
    )
)

In [ ]:
planificaciones = (
    df_movimientos.loc[
        (df_movimientos["TipoVía"] == "PLANNED")
        & (df_movimientos["Retraso"] == 0)
        & np.invert(
            df_movimientos["Producto"].isin([" ", "Material Vacio", "Servicio Interno"])
        ),
        [
            "NTécnico",
            "FechaOrigenTécnico",
            "NComercial",
            "FechaOrigenComercial",
            # "Código",
            # "Nombre",
            "CódigoOrigen",
            "NombreOrigen",
            "CódigoDestino",
            "NombreDestino",
            "Línea",
            "Tráfico",
            "Operador",
            "Producto",
            # "TipoParada",
            "TiempoPlanificado",
            # "Retraso",
            "VíaPlanificada",
            "VíaSitra",
            "VíaCTC",
            # "TipoVía",
            # "Estado",
        ],
    ]
    .dropna(subset=["NTécnico"])
    .sort_values(by="TiempoPlanificado")
    .reset_index(drop=True)
)
planificaciones[
    [
        "EN APP?",
        "VIA APP",
        "VIA MONITOR",
        "HORA REAL ENTRA",
        "VIA REAL",
        "COMENTARIOS HORA APP",
        "COMENTARIOS HORA MONITOR",
    ]
] = None
cercanias = planificaciones[
    (planificaciones["Producto"] == "CERCANIAS")
    & (
        planificaciones["VíaPlanificada"].apply(
            lambda x: int(regex.search(r"\d+", x).group())
        )
        < 13
    )
].drop(
    ["NComercial", "FechaOrigenComercial", "Tráfico", "Producto", "Operador"], axis=1
)
no_cercanias = planificaciones[
    (planificaciones["Tráfico"] == "AVLDMD")
    & np.invert(planificaciones["Producto"].isin(["CERCANIAS"]))
    & (
        planificaciones["VíaPlanificada"].apply(
            lambda x: int(regex.search(r"\d+", x).group())
        )
        < 13
    )
].drop(["Tráfico", "Operador", "Línea"], axis=1)
alta_velocidad = planificaciones[
    (planificaciones["Tráfico"] == "AVLDMD")
    & np.invert(
        planificaciones["Producto"].isin(
            ["CERCANIAS", "REGIONAL", "REGIONAL EXPRES", "MD"]
        )
    )
    & (
        planificaciones["VíaPlanificada"].apply(
            lambda x: int(regex.search(r"\d+", x).group())
        )
        > 13
    )
].drop(["Tráfico", "Operador", "Línea"], axis=1)

In [ ]:
# guardarExcel(
#     cercanias[
#         (cercanias["TiempoPlanificado"] > pd.to_datetime("2024-11-19 09:30:00"))
#         & (cercanias["TiempoPlanificado"] < pd.to_datetime("2024-11-19 11:30:00"))
#     ],
#     "Revisión APP - Chamartín.xlsx",
#     sheet_name="Cercanías",
#     append_sheet=False,
# )
# guardarExcel(
#     no_cercanias[
#         (no_cercanias["TiempoPlanificado"] > pd.to_datetime("2024-11-19 11:00:00"))
#         & (no_cercanias["TiempoPlanificado"] < pd.to_datetime("2024-11-19 13:00:00"))
#     ],
#     "Revisión APP - Chamartín.xlsx",
#     sheet_name="ConvencionalNoCercanías",
#     append_sheet=True,
# )
# guardarExcel(
#     alta_velocidad[
#         (alta_velocidad["TiempoPlanificado"] > pd.to_datetime("2024-11-19 12:30:00"))
#         & (alta_velocidad["TiempoPlanificado"] < pd.to_datetime("2024-11-19 14:30:00"))
#     ],
#     "Revisión APP - Chamartín.xlsx",
#     sheet_name="AV",
#     append_sheet=True,
# )

In [ ]:
import plotly.graph_objects as go

fig = go.Figure()
fig.add_trace(
    go.Histogram(x=df_movimientos["TiempoPlanificado"], opacity=0.75, nbinsx=96)
)

fig.update_layout(
    title_text="Sampled Results",  # title of plot
    xaxis_title_text="Value",  # xaxis label
    yaxis_title_text="Count",  # yaxis label
    bargap=0.01,  # gap between bars of adjacent location coordinates
    bargroupgap=0.01,  # gap between bars of the same location coordinates
)

fig.show()

### Estaciones

In [ ]:
HOSTPATH = "/portroyalmanager/stations/allstations/"
data = {
    "detailedInfo": {
        "extendedStationInfo": True,
        "stationActivities": True,
        "stationBanner": True,
        "stationCommercialServices": True,
        "stationInfo": True,
        "stationServices": True,
        "stationTransportServices": True,
    },
    "filter": "ALL",
    "token": "string",
}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    URL + HOSTPATH,
    headers=headers,
    data=data,
)
info_estaciones = [
    json.loads(el.strip("data:")) for el in (regex.split(r"\n+", response.text.strip()))
]
# info_estaciones = pd.json_normalize(
#     [el["requestedStationInfo"] for el in info_estaciones]
# )

In [ ]:
a = pd.json_normalize(info_estaciones)
a[a["requestedStationInfo.stationCode"] == "17000"].values

In [ ]:
# info_estaciones = info_estaciones[
#     [
#         "stationCode",
#         "stationInfo.stationType",
#         "stationInfo.longName",
#         "stationInfo.shortName",
#         "stationInfo.trafficType",
#         "stationInfo.commuterNetwork",
#         "stationInfo.lines",
#         "stationInfo.location.longitude",
#         "stationInfo.location.latitude",
#     ]
# ].rename(
#     columns={
#         "stationCode": "Código",
#         "stationInfo.stationType": "Tipo",
#         "stationInfo.longName": "Nombre",
#         "stationInfo.shortName": "NombreCorto",
#         "stationInfo.trafficType": "Tráfico",
#         "stationInfo.commuterNetwork": "Cercanías",
#         "stationInfo.lines": "Lineas",
#         "stationInfo.location.longitude": "Longitud",
#         "stationInfo.location.latitude": "Latitud",
#     }
# )
# info_estaciones = info_estaciones.sort_values(by=["Tipo", "Código"])
# # guardarExcel(info_estaciones, "data/info_estaciones.xlsx")

In [ ]:
info_estaciones

In [ ]:
# info_estaciones[info_estaciones["Código"] == "17000"]

In [ ]:
headers

In [ ]:
HOSTPATH = "/portroyalmanager/stations/filtered/"
data = {
    "detailedInfo": {
        "extendedStationInfo": True,
        "stationActivities": True,
        "stationBanner": True,
        "stationCommercialServices": True,
        "stationInfo": True,
        "stationServices": True,
        "stationTransportServices": True,
    },
    "filter": "ALL",
    "token": "string",
}
data = json.dumps(data)
response = requests.request(
    "POST",
    URL + HOSTPATH,
    headers=headers,
    data=data,
)

In [ ]:
response

In [ ]:
HOSTPATH = "/portroyalmanager/stations/filtered/"
data = {
    "detailedInfo": {
        "extendedStationInfo": True,
        "stationActivities": True,
        "stationBanner": True,
        "stationCommercialServices": True,
        "stationInfo": True,
        "stationServices": True,
        "stationTransportServices": True,
    },
    "filter": "ALL",
    "token": "string",
}
data = json.dumps(data)

response = hacerPeticion(
    "POST",
    URL + HOSTPATH,
    headers=headers,
    data=data,
)
info_estaciones_filtered = [
    json.loads(el.strip("data:")) for el in (regex.split(r"\n+", response.text.strip()))
]

In [ ]:
info_estaciones = pd.json_normalize(
    [el["requestedStationInfo"] for el in info_estaciones_filtered]
)
rename_cols = {
    "stationCode": "Código",
    "stationInfo.stationType": "Tipo",
    "stationInfo.longName": "Nombre",
    "stationInfo.shortName": "NombreCorto",
    "stationInfo.trafficType": "Tráfico",
    "stationInfo.commuterNetwork": "Cercanías",
    "stationInfo.lines": "Lineas",
    "stationInfo.location.longitude": "Longitud",
    "stationInfo.location.latitude": "Latitud",
}
info_estaciones = info_estaciones[list(rename_cols.keys())].rename(columns=rename_cols)
info_estaciones = info_estaciones.sort_values(by=["Tipo", "Código"])

In [ ]:
guardarExcel(info_estaciones, "data/info_estaciones.xlsx", append_sheet=False)

### Histórico MOW

In [ ]:
HOSTPATH = "/movements/history/mow/"
inicio = "2024-10-27"
fin = "2024-10-27"

movimientos = []
page = 0
while True:
    print(f"Página {page}")
    data = {
        "page": page,
        "size": 1000,
        "initDate": int(pd.to_datetime(inicio).timestamp()) * 1000,
        "endDate": int(pd.to_datetime(fin).timestamp()) * 1000,
        "technicalNumbers": [],
        "stationCodes": ["17000"],
    }
    data = json.dumps(data)

    response = hacerPeticion(
        "POST",
        URL + HOSTPATH,
        headers=headers,
        data=data,
    )
    if not response:
        break
    response = response.json()
    movimientos.extend(response["movementList"]["list"])
    page += 1
    if page == response["totalPages"]:
        break

In [ ]:
df = pd.DataFrame(movimientos)
rename_cols = {
    "datetime": "Fecha",
    "technicalNumber": "NTécnico",
    "stationName": "Nombre",
    "stationCode": "Código",
    "sequence": "Secuencia",
    "movementType": "Movimiento",
    "movementSource": "FuenteMovimiento",
    "platform": "Vía",
    "sourcePlatform": "FuenteVía",
    "delayInSeconds": "Retraso_s",
    "operator": "Operador",
    "product": "Producto",
    "trainCompanyCode": "Empresa",
    "triggerElement": "ElementoDisparo",
    "originDate": "FechaOrigen",
}
df[list(rename_cols.keys())].rename(columns=rename_cols)
# df["datetime"].apply(time2localtime, unit="ms")
# df["originDate"].apply(time2localtime, unit="ms")
# df